# Faithfulness Under Retrieval Failure: RAG for Low-Resource Urdu QA

This notebook asks one focused question in a careful way: **when the "search" part of a
question-answering system fails — hands the model the wrong document, or nothing useful — does the
model start making things up *more* than if it had no document at all? And can we catch that
automatically?**

You don't need a heavy background. The short version:

- A popular way to build a QA system is **RAG** = *Retrieval-Augmented Generation*. Two steps:
  first a **retriever** searches a pile of documents for ones relevant to your question, then a
  **generator** (a language model) reads those documents and writes an answer. The idea is the
  model answers *from real documents* instead of from memory, so it makes fewer things up.
- But retrieval isn't perfect, especially in a **low-resource language** like Urdu where the tools
  are weaker. So what happens when the retriever screws up and feeds the model an *irrelevant*
  document? Our worry: the model treats that wrong document as gospel and confidently answers
  wrong — possibly *worse* than if we'd given it no document and let it just say "I don't know."

That "confident wrong answer" behavior is called **hallucination**, and this project measures it.

**The specific bet we're testing (our hypothesis — could be wrong, that's fine):**
RAG in a low-resource language hallucinates *more* under retrieval failure than a plain no-retrieval
baseline, because the model over-trusts whatever text it's handed.

**What this notebook does, same style as the sibling HE-vision project:**
- runs entirely on **free Colab**, no paid API keys anywhere,
- **saves everything to Google Drive**, and on later runs **loads instead of recomputing** the slow
  parts,
- explains every step in plain words,
- builds its own **adversarial "broken retrieval" test set** automatically, so there's no manual
  labeling required to get a result (though you can add your own examples to strengthen it).


## 1. Setup

Installing tools and picking a GPU if one's available. Plumbing — nothing to understand here.

In [ ]:
# sentence-transformers = the retriever (turns text into searchable vectors)
# transformers + torch = the generator (the language model that writes answers)
# the rest are helpers. everything here is free and open.
!pip install -q torch transformers sentence-transformers scikit-learn pandas matplotlib tqdm accelerate

In [ ]:
import os, re, json, random, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
import matplotlib.pyplot as plt

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("running on:", DEVICE)

## 2. Connect Google Drive (save results, and reload them on later runs)

Same trick as the vision project. We mount Drive, pick one folder, and save everything there —
the built test set, the model's answers, the scores. The slow parts (embedding the documents,
generating answers) check Drive first and **reload instead of redoing the work** if they've already
run once. First run is the slow one; everything after is fast.

Not on Colab? The `try/except` quietly falls back to a local folder so it still runs.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/RAG_Urdu_Faithfulness"
except Exception:
    print("not on colab — saving locally instead.")
    SAVE_DIR = "./RAG_Urdu_Faithfulness"
os.makedirs(SAVE_DIR, exist_ok=True)
print("saving everything to:", SAVE_DIR)

def path(name): return os.path.join(SAVE_DIR, name)
def save_json(obj, name):
    with open(path(name), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
def load_json(name):
    with open(path(name), encoding="utf-8") as f:
        return json.load(f)
def exists(name): return os.path.exists(path(name))

# flip to True to wipe cached results and redo everything from scratch
FORCE_REDO = False

## 3. The knowledge base + questions (real Urdu dataset, with a safety net)

This loads a **real, public Urdu QA dataset** from HuggingFace. We try a couple of good ones in
order and use the first that loads:

1. **UQA** (`uqa`) — a large Urdu QA set built by translating SQuAD2.0. SQuAD-style
   `context / question / answer`, which is exactly the shape a RAG study needs.
2. **LEGAL-UQA** — Urdu QA over Pakistan's constitution (ties nicely to the "Pakistani law" angle).
3. If neither is reachable on the day (HuggingFace hiccups, offline, etc.), we **fall back to a small
   built-in Urdu set** so the notebook *always* runs and never hard-fails on you.

You control the size with `MAX_ITEMS` — keep it modest (a few hundred) so generation stays fast on
free Colab. Each item ends up as: a `question`, the `gold_answer`, and the `gold_doc` (the passage
that actually contains the answer). Later we deliberately feed the model the *wrong* passage to see
what it does.

Plain-language note on why real data matters here: with the tiny built-in set the hallucination
numbers are noisy and only demonstrate the *method*. A real dataset of a few hundred questions is
what turns this into an actual result worth putting in the README.

In [ ]:
!pip install -q datasets bitsandbytes

In [ ]:
from datasets import load_dataset

MAX_ITEMS = 200   # keep modest so generation stays fast on free colab. bump up if you have time.

# small built-in set used ONLY if every online dataset fails to load, so the notebook never dies.
BUILTIN = [
    {"question": "پاکستان کا دارالحکومت کیا ہے؟", "gold_answer": "اسلام آباد",
     "gold_doc": "اسلام آباد پاکستان کا دارالحکومت ہے اور یہ ملک کا دسواں بڑا شہر ہے۔"},
    {"question": "قائد اعظم کا پورا نام کیا تھا؟", "gold_answer": "محمد علی جناح",
     "gold_doc": "قائد اعظم محمد علی جناح پاکستان کے بانی تھے اور گیارہ ستمبر انیس سو اڑتالیس کو وفات پائی۔"},
    {"question": "دریائے سندھ کہاں سے نکلتا ہے؟", "gold_answer": "تبت",
     "gold_doc": "دریائے سندھ تبت کے سطح مرتفع سے نکلتا ہے اور پاکستان سے ہوتا ہوا بحیرہ عرب میں گرتا ہے۔"},
    {"question": "پاکستان کی قومی زبان کیا ہے؟", "gold_answer": "اردو",
     "gold_doc": "اردو پاکستان کی قومی زبان ہے جبکہ انگریزی دفتری زبان ہے۔"},
    {"question": "کے ٹو پہاڑ کہاں واقع ہے؟", "gold_answer": "گلگت بلتستان",
     "gold_doc": "کے ٹو دنیا کی دوسری بلند ترین چوٹی ہے اور گلگت بلتستان میں واقع ہے۔"},
]

def _row_to_item(context, question, answer):
    """keep only clean, answerable rows: need a non-empty context, question, and answer,
    the answer must actually appear in the context (so 'gold_doc' really is the gold),
    AND the answer must be SHORT (a factoid, not a long span) — long spans are impossible
    for a small model to reproduce, which quietly zeroes out every score. this filter is
    the difference between a real result and a floor of zeros."""
    if not context or not question or not answer:
        return None
    context = str(context).strip(); question = str(question).strip(); answer = str(answer).strip()
    if len(answer) < 1 or answer not in context:
        return None
    # short-answer only: at most 4 words and 40 chars. keeps the task fair for a small model.
    if len(answer.split()) > 4 or len(answer) > 40:
        return None
    return {"question": question, "gold_answer": answer, "gold_doc": context}

def _answer_from_row(r):
    """UQA/SQuAD-style rows keep the answer under answers.text[0]; some mirrors use 'answer'."""
    ans = r.get("answers")
    if isinstance(ans, dict):
        t = ans.get("text")
        if isinstance(t, list) and t: return t[0]
        if isinstance(t, str): return t
    for key in ("answer", "answer_text", "gold_answer"):
        if r.get(key): return r[key]
    return None

def _harvest(ds):
    items = []
    for r in ds:
        it = _row_to_item(r.get("context"), r.get("question"), _answer_from_row(r))
        if it: items.append(it)
        if len(items) >= MAX_ITEMS: break
    return items

# UQA lives under a user namespace on the Hub. exact owner can drift, so we try a few
# known-good spellings and use whichever resolves. if you know the exact id, put it first.
UQA_IDS = ["sameearif/uqa", "sameearif/UQA", "SameeArif/UQA", "uqa"]

def try_uqa():
    last = None
    for ds_id in UQA_IDS:
        for split in ("validation", "test", "train"):
            try:
                ds = load_dataset(ds_id, split=split)
                got = _harvest(ds)
                if got and len(got) >= 10:
                    print(f"   -> loaded from '{ds_id}' [{split}]")
                    return got
            except Exception as e:
                last = e
    if last: raise last
    return []

# LEGAL-UQA: Urdu QA over Pakistan's constitution. ties to the 'Pakistani law' angle.
# it's a generative-QA set (answers may be longer/abstractive), so the short-answer filter in
# _row_to_item will keep only the crisp factoid rows — which is exactly what we want here.
LEGAL_IDS = ["mbshm/legal-uqa", "sameearif/legal-uqa", "SameeArif/LEGAL-UQA"]

def try_legal_uqa():
    last = None
    for ds_id in LEGAL_IDS:
        for split in ("train", "validation", "test"):
            try:
                ds = load_dataset(ds_id, split=split)
                # legal-uqa columns vary by mirror; probe a few likely names
                items = []
                for r in ds:
                    ctx = r.get("context") or r.get("urdu_context") or r.get("Context")
                    q   = r.get("question") or r.get("urdu_question") or r.get("Question")
                    a   = _answer_from_row(r) or r.get("urdu_answer") or r.get("Answer")
                    it = _row_to_item(ctx, q, a)
                    if it: items.append(it)
                    if len(items) >= MAX_ITEMS: break
                if items and len(items) >= 10:
                    print(f"   -> loaded from '{ds_id}' [{split}]")
                    return items
            except Exception as e:
                last = e
    if last: raise last
    return []

# try UQA first (bigger, cleaner factoids), then LEGAL-UQA, then fall back to built-in.
SOURCES = [("UQA", try_uqa), ("LEGAL-UQA", try_legal_uqa)]

KB = None
for name, fn in SOURCES:
    try:
        print(f"trying to load {name} ...")
        cand = fn()
        if cand and len(cand) >= 10:
            KB = cand
            print(f"loaded {len(KB)} items from {name}")
            break
        print(f"{name} returned too few usable rows, trying next.")
    except Exception as e:
        print(f"{name} failed ({type(e).__name__}), trying next.")

USING_REAL_DATA = KB is not None
if KB is None:
    print("all online datasets unavailable — using the small built-in set so the notebook still runs.")
    KB = BUILTIN

print("\n" + ("="*60))
if USING_REAL_DATA:
    print("STATUS: running on a REAL Urdu dataset. numbers here are report-worthy.")
else:
    print("STATUS: running on the tiny BUILT-IN set (fallback). numbers are a")
    print("        method demo only — rerun when the Hub is reachable for real results.")
print("="*60)

# de-duplicate passages for the corpus (many QA rows can share a context)
CORPUS = list(dict.fromkeys(item["gold_doc"] for item in KB))
DOC2IDX = {d: i for i, d in enumerate(CORPUS)}
print(f"\n{len(KB)} question/answer pairs | {len(CORPUS)} unique passages in the knowledge base")
print("example question:", KB[0]["question"][:70])
print("example answer  :", KB[0]["gold_answer"][:40])

## 4. The retriever (the "search" step)

The retriever turns every passage into a list of numbers (an **embedding**) that captures its
meaning, so we can find the passage whose meaning is closest to the question. We use a **multilingual**
embedding model so it handles Urdu.

We embed the whole corpus once and cache it to Drive — this is one of the slow steps that gets
skipped on later runs.

In [ ]:
from sentence_transformers import SentenceTransformer

# multilingual model so Urdu works. small enough for free Colab.
retriever = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                                device=DEVICE)

if exists("corpus_emb.npy") and not FORCE_REDO:
    corpus_emb = np.load(path("corpus_emb.npy"))
    print("loaded cached corpus embeddings (skipped re-embedding)")
else:
    corpus_emb = retriever.encode(CORPUS, convert_to_numpy=True, normalize_embeddings=True)
    np.save(path("corpus_emb.npy"), corpus_emb)
    print("embedded corpus and cached it")

def retrieve(question, k=1):
    q = retriever.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    sims = (corpus_emb @ q.T).ravel()           # cosine similarity (higher = more relevant)
    idx = sims.argsort()[::-1][:k]              # top-k most similar passages
    return [(int(i), CORPUS[i], float(sims[i])) for i in idx]

# quick sanity check
demo = retrieve(KB[0]["question"], k=1)[0]
print("Q:", KB[0]["question"])
print("top retrieved passage:", demo[1][:60], "...")

## 5. The generator (the language model that writes the answer)

This reads the question (plus whatever passage we hand it) and writes a short Urdu answer. We use a
mid-size open model (**Qwen2.5-7B-Instruct**) loaded in **4-bit**, which fits on a free Colab T4 and
is — importantly — big enough to make a real judgment call about whether it actually knows something.
(A tiny 1.5B model can't abstain sensibly; it either answers everything or nothing. If you somehow
run without a GPU, the code quietly drops to the 1.5B model so it still works.)

The prompt is deliberately **balanced**: answer in a few words *if the answer is there*, otherwise
say "معلوم نہیں" (I don't know). This matters for the whole study. If we force abstention the model
never answers; if we forbid it the model never abstains — either way there's nothing to measure.
Giving a genuine *option* to abstain is what makes abstention a real signal, so we can actually tell
"confidently wrong" apart from "honestly unsure." That distinction is the core of the whole project.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# bigger model = actually capable of judging "do I know this or not?". a 1.5B model can't abstain
# sensibly. 4-bit quantization keeps the 7B inside a free T4's memory.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

if DEVICE == "cuda":
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16)
    gen_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                                     device_map="auto").eval()
else:
    # cpu fallback: drop to the small model so it still runs without a gpu
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    gen_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).eval()

@torch.no_grad()
def generate_answer(question, context=None):
    """balanced prompt: answer in a few words IF you know, otherwise say 'معلوم نہیں'.
    the key is 'if you actually know' — we give a real option to abstain without commanding it,
    so abstention becomes a genuine signal instead of always-on or always-off."""
    if context:
        user = (f"نیچے دیا گیا متن پڑھیں۔ اگر متن میں سوال کا جواب موجود ہے تو صرف چند الفاظ میں جواب دیں۔ "
                f"اگر متن میں جواب موجود نہیں تو صرف 'معلوم نہیں' لکھیں۔\n\n"
                f"متن: {context}\n\nسوال: {question}\nجواب:")
    else:
        user = (f"سوال کا جواب صرف چند الفاظ میں دیں۔ اگر آپ واقعی جواب نہیں جانتے تو 'معلوم نہیں' لکھیں۔\n\n"
                f"سوال: {question}\nجواب:")
    msgs = [{"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(gen_model.device)
    out = gen_model.generate(**inp, max_new_tokens=24, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return ans.strip()

print("generator ready.")
print("test:", generate_answer(KB[0]["question"], KB[0]["gold_doc"]))

## 6. Build the "broken retrieval" test set (the heart of the project)

This is where we deliberately sabotage retrieval to study what the model does. For each question we
create **three conditions**:

1. **correct** — the model gets the *right* passage (retrieval worked). Best case.
2. **corrupted** — the model gets a *wrong, unrelated* passage (retrieval failed but still handed
   over something). This is the dangerous case we care about.
3. **none** — the model gets *no* passage and must answer from memory (no retrieval at all).

Our hypothesis says condition 2 (corrupted) produces *more* confident-but-wrong answers than
condition 3 (none). By building all three from the same questions, it's a clean, controlled
comparison.

The set is built automatically — no manual labeling needed. You *can* append your own tricky
examples to make it stronger; there's a marked spot for that.

In [ ]:
def build_eval_set():
    rows = []
    for i, item in enumerate(KB):
        q, gold, gold_doc = item["question"], item["gold_answer"], item["gold_doc"]
        # a WRONG passage = any corpus passage that does NOT contain this answer.
        # (checking the answer isn't in it makes sure "corrupted" is genuinely unhelpful.)
        wrong_pool = [d for d in CORPUS if gold not in d]
        if not wrong_pool:            # tiny-corpus safety
            wrong_pool = [d for d in CORPUS if d != gold_doc] or [gold_doc]
        wrong_doc = random.choice(wrong_pool)
        rows.append({"qid": i, "question": q, "gold_answer": gold, "condition": "correct", "context": gold_doc})
        rows.append({"qid": i, "question": q, "gold_answer": gold, "condition": "corrupted", "context": wrong_doc})
        rows.append({"qid": i, "question": q, "gold_answer": gold, "condition": "none", "context": None})
    return rows

# --- ADD-YOUR-OWN spot: append extra hand-written {question, gold_answer, context, condition} dicts here ---
EXTRA = []

eval_set = build_eval_set() + EXTRA
save_json(eval_set, "eval_set.json")
print(f"built {len(eval_set)} test cases ({len(KB)} questions x 3 conditions)")

## 7. Run the model on every condition

We ask the model all three versions of each question and record its answers. This is the other slow
step, so results get cached to Drive and reloaded next time.

In [ ]:
def norm(s):
    # loose text cleanup so "اسلام آباد۔" matches "اسلام آباد"
    return re.sub(r"[\s۔.,!؟?\-]+", " ", str(s)).strip()

def is_correct(pred, gold):
    """forgiving match: right if the gold answer's words show up in the prediction.
    a small model paraphrases, so demanding an exact string match unfairly scores everything
    wrong. we require the gold tokens to be present (order-free) — close enough for a factoid."""
    p = norm(pred); g = norm(gold)
    if g and g in p:                       # direct hit
        return True
    gold_toks = [t for t in g.split() if len(t) > 1]
    if not gold_toks:
        return g in p
    hits = sum(1 for t in gold_toks if t in p)
    return hits / len(gold_toks) >= 0.6    # most of the gold answer's words appear

def looks_like_idk(pred):
    # did the model refuse / hedge instead of committing to an answer?
    p = norm(pred)
    return ("معلوم نہیں" in p or "نہیں معلوم" in p or "علم نہیں" in p
            or "جواب نہیں" in p or len(p) == 0)

if exists("answers.json") and not FORCE_REDO:
    answers = load_json("answers.json")
    print("loaded cached answers (skipped re-generation)")
else:
    answers = []
    for r in eval_set:
        pred = generate_answer(r["question"], r["context"])
        answers.append({**r,
                        "pred": pred,
                        "correct": is_correct(pred, r["gold_answer"]),
                        "abstained": looks_like_idk(pred)})
    save_json(answers, "answers.json")
    print("generated all answers and cached them")

ans_df = pd.DataFrame(answers)
print(ans_df[["condition", "correct", "abstained"]].groupby("condition").mean())

## 8. The main result: does broken retrieval cause more hallucination?

We define a **hallucination** as: the model gave a confident answer (didn't say "I don't know") and
it was **wrong**. Now we compare hallucination rates across the three conditions.

What we're looking for:
- if **corrupted** has a *higher* hallucination rate than **none**, our hypothesis holds — a wrong
  document is worse than no document, because the model over-trusts it.
- if they're similar, the model is ignoring bad context (good for it, mild for our hypothesis).
- if corrupted is *lower*, that would be surprising and worth a hard look.

In [ ]:
def hallucination_rate(df_sub):
    # confident (didn't abstain) AND wrong
    confident_wrong = (~df_sub["abstained"]) & (~df_sub["correct"])
    return confident_wrong.mean()

summary = {}
for cond in ["correct", "corrupted", "none"]:
    sub = ans_df[ans_df["condition"] == cond]
    summary[cond] = {
        "accuracy": round(sub["correct"].mean(), 3),
        "abstain_rate": round(sub["abstained"].mean(), 3),
        "hallucination_rate": round(hallucination_rate(sub), 3),
    }

summary_df = pd.DataFrame(summary).T
print(summary_df)
save_json(summary, "condition_summary.json")

h_corr = summary["corrupted"]["hallucination_rate"]
h_none = summary["none"]["hallucination_rate"]
print(f"\ncorrupted hallucination rate: {h_corr:.3f}")
print(f"none      hallucination rate: {h_none:.3f}")
if h_corr > h_none + 0.05:
    print("=> hypothesis SUPPORTED: a wrong document made the model hallucinate MORE than no document.")
elif h_corr < h_none - 0.05:
    print("=> hypothesis REVERSED (and this is the interesting result): a wrong document made the")
    print("   model hallucinate LESS than no document. check the abstain rates above — most likely")
    print("   the model NOTICES the wrong passage doesn't fit and abstains, instead of trusting it.")
    print("   that's a robustness finding: with a capable model, bad retrieval mostly causes honest")
    print("   'I don't know's, not confident lies. worth reporting exactly as-is.")
else:
    print("=> roughly equal: the model handled a wrong document about as (un)helpfully as none.")

In [ ]:
plt.figure(figsize=(7, 4))
conds = ["correct", "corrupted", "none"]
plt.bar(conds, [summary[c]["hallucination_rate"] for c in conds])
plt.ylabel("hallucination rate (confident + wrong)")
plt.title("Does broken retrieval cause more made-up answers?")
plt.tight_layout(); plt.savefig(path("hallucination_by_condition.png"), dpi=120); plt.show()

## 9. Can we catch wrong answers automatically?

A practical follow-up: can we **flag** likely-wrong answers so a downstream system knows not to
trust them? We test what signals actually work.

The honest result up front: **retrieval similarity alone barely helps** here (you'll see an AUROC
near 0.5, which means useless). That itself is a finding — the intuition "low similarity = wrong
answer" didn't hold. What *does* carry signal is the model's **own abstention**: when this model
says "معلوم نہیں," it's usually right to be unsure. So we test both, separately and combined, and
report what wins instead of pretending our first guess worked.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

# assemble features for each answer
rows = []
for r in answers:
    top = retrieve(r["question"], k=1)[0]
    rows.append({
        "retrieval_sim": top[2],            # how relevant the best passage looked
        "abstained": int(r["abstained"]),   # did the model hedge?
        "is_wrong": int(not r["correct"]),  # what we're trying to predict
    })
feat_df = pd.DataFrame(rows)
save_json(feat_df.to_dict(orient="records"), "detector_features.json")

def safe_auc(y, score):
    return roc_auc_score(y, score) if len(set(y)) > 1 else float("nan")

y = feat_df["is_wrong"].values

# signal 1: retrieval similarity alone (lower sim -> guess wrong). our original idea.
auc_sim = safe_auc(y, -feat_df["retrieval_sim"].values)

# signal 2: abstention alone (abstained -> guess wrong, since abstain answers score as wrong).
auc_abs = safe_auc(y, feat_df["abstained"].values)

# signal 3: both together in a tiny logistic-regression detector, honestly cross-validated.
X = feat_df[["retrieval_sim", "abstained"]].values
if len(set(y)) > 1:
    proba = cross_val_predict(LogisticRegression(max_iter=1000), X, y, cv=5,
                              method="predict_proba")[:, 1]
    auc_combined = safe_auc(y, proba)
else:
    auc_combined = float("nan")

print("how well each signal predicts a wrong answer (AUROC; 0.5 = useless, 1.0 = perfect):")
print(f"  retrieval similarity alone : {auc_sim:.3f}")
print(f"  model's abstention alone   : {auc_abs:.3f}")
print(f"  both combined (logistic)   : {auc_combined:.3f}")
print("\ntakeaway: whichever is highest is the signal worth using. if abstention wins, the model")
print("knowing its own limits is a better wrongness-detector than second-guessing the retriever.")
save_json({"auc_similarity": auc_sim, "auc_abstention": auc_abs, "auc_combined": auc_combined},
          "detector_scores.json")

## 10. Save a tidy summary to Drive

In [ ]:
final = {
    "n_questions": len(KB),
    "per_condition": summary,
    "hypothesis_supported": bool(summary["corrupted"]["hallucination_rate"] > summary["none"]["hallucination_rate"]),
    "note": "small built-in KB; swap in Urdu Wikipedia / UQuAD (section 3) for headline numbers.",
}
save_json(final, "SUMMARY.json")
print(json.dumps(final, ensure_ascii=False, indent=2))

## 11. What we did, in one breath

We built a small Urdu question-answering system, then deliberately broke its "search" step three
different ways — right passage, wrong passage, no passage — and measured whether feeding it a *wrong*
passage makes it confidently make things up more than giving it *nothing*. Then we built a cheap
detector that uses retrieval similarity to flag likely-wrong answers.

Everything's cached to Drive, so re-running is fast. The one thing to do for headline-worthy numbers
is swap the tiny built-in knowledge base for a real Urdu corpus at the marked spot in section 3 —
the rest of the notebook doesn't change.

**Honest scope note:** with the tiny built-in set, treat the numbers as a *working demonstration of
the method*, not a finished finding. The method is the deliverable; the big Urdu corpus is what turns
it into a result worth writing up.